In [ ]:
import geopandas as gpd
import pandas as pd
from sympy import false

from Himalaya_greatGlacier.merge_thick_baredem import working_dir

# 读取数据
lia = gpd.read_file("/mnt/e/2Data/022_工作2/出图数据/Himalaya_LIA_with_RGIID_Complete/Himalaya_LIA_with_RGIID_Complete.shp")
rgi = gpd.read_file("/mnt/e/2Data/022_工作2/出图数据/himalaya_glaciers_complete/himalaya_glaciers_complete.shp")

# 1. 筛选 RGI 面积 >= 10 km²
rgi_large = rgi[rgi['Area'] >= 10].copy()
large_rgi_ids = set(rgi_large['RGIId'])

# 2. 找包含这些大冰川的 LIA
def contains_large_rgi(rgi_str):
    if pd.isna(rgi_str):
        return False
    ids = [x.strip() for x in str(rgi_str).split(';')]
    return bool(set(ids) & large_rgi_ids)

lia_matched = lia[lia['AllRGIs'].apply(contains_large_rgi)].copy()

# 3. 提取这些 LIA 涉及的所有 RGI ID
all_related_rgi_ids = set()
for rgi_str in lia_matched['AllRGIs'].dropna():
    ids = [x.strip() for x in str(rgi_str).split(';')]
    all_related_rgi_ids.update(ids)

# 4. 筛选对应的所有现代冰川
rgi_matched = rgi[rgi['RGIId'].isin(all_related_rgi_ids)].copy()

print(f"RGI >= 10 km²: {len(rgi_large)} 个")
print(f"包含大冰川的 LIA: {len(lia_matched)} 个")
print(f"这些 LIA 涉及的所有 RGI: {len(rgi_matched)} 个")

# 可选：保存结果
# lia_matched.to_file("Him_10_lia_with_large.shp")
# rgi_matched.to_file("Him_10_rgi_related_lia.shp")

In [ ]:
from oggm import cfg, workflow, tasks

cfg.initialize()
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_multiprocessing'] = True
cfg.PATHS['working_dir'] = '/mnt/f/oggm_related/oggm_rgi_calibration'

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd

os.environ['http_proxy'] = 'http://192.168.160.1:7890'
os.environ['https_proxy'] = 'http://192.168.160.1:7890'

from oggm import cfg, workflow, tasks, utils

# 工作目录
WORKING_DIR = "/mnt/f/oggm_related/oggm_LIA_simulation"

# 床基DEM
DEM_PATH = "/mnt/f/oggm_related/oggm_LIA_himalaya_great/output/Himalaya_bare_DEM.tif"

# 准备LIA编目
lia_for_oggm = lia_matched.copy()
lia_for_oggm = lia_for_oggm.to_crs("EPSG:4326")

# cook_rgidf自动补全所有必需字段（包括RGIId）
lia_prepared = utils.cook_rgidf(lia_for_oggm, o1_region='15')

print(f"准备完成，共 {len(lia_prepared)} 个冰川")
print(f"字段: {lia_prepared.columns.tolist()}")

# OGGM配置
cfg.initialize(logging_level='WARNING')
cfg.PATHS['working_dir'] = WORKING_DIR
cfg.PATHS['dem_file'] = DEM_PATH
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20

# 初始化
# gdirs = workflow.init_glacier_directories(lia_prepared, reset=True)
gdirs = workflow.init_glacier_directories(lia_prepared)

In [ ]:
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['use_intersects'] = False
cfg.PATHS['dem_file'] = DEM_PATH
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['use_multiple_flowlines'] = True
cfg.PARAMS['use_multiprocessing'] = True

for gdir in gdirs:
    tasks.define_glacier_region(gdir, source='USER')


In [ ]:
from oggm.workflow import execute_entity_task
cfg.PARAMS['use_multiprocessing'] = True
task_list = [
    tasks.glacier_masks,
    tasks.compute_centerlines,
    tasks.initialize_flowlines,
    tasks.compute_downstream_line,
    tasks.compute_downstream_bedshape,
    tasks.catchment_area,
    tasks.catchment_intersections,
    tasks.catchment_width_geom,
    tasks.catchment_width_correction,
]
for task in task_list:
    execute_entity_task(task, gdirs)


In [ ]:
cfg.PARAMS['baseline_climate'] = 'CRU'  # 清空，使用默认

for gdir in gdirs:
    tasks.process_cru_data(gdir)
#
# tasks.process_custom_climate_data

In [ ]:
from oggm.core.massbalance import apparent_mb_from_linear_mb

workflow.execute_entity_task(apparent_mb_from_linear_mb, gdirs)

In [ ]:
execute_entity_task(tasks.apparent_mb_from_any_mb, gdirs)
execute_entity_task(tasks.prepare_for_inversion, gdirs)
execute_entity_task(tasks.mass_conservation_inversion, gdirs)
execute_entity_task(tasks.filter_inversion_output, gdirs)

# 初始化动力学模型
execute_entity_task(tasks.init_present_time_glacier, gdirs)

In [ ]:
from oggm.shop import gcm_climate

# 处理LMR气候数据（会自动下载）
workflow.execute_entity_task(gcm_climate.process_lmr_data, gdirs,
                            output_filesuffix='_LMR')

In [ ]:
success_spinup = []
fail_spinup = []
from oggm.core.flowline import run_constant_climate, run_from_climate_data

for gdir in gdirs:
    try:
        run_constant_climate(
            gdir,
            nyears=300,           # 足够长让冰川达到平衡
            y0=1650,              # LIA冷期中心年份
            halfsize=30,          # 用1620-1680的30年平均气候
            climate_filename='gcm_data',
            climate_input_filesuffix='_LMR',
            output_filesuffix='_spinup_LIA',
            store_model_geometry=True,
        )
        success_spinup.append(gdir.rgi_id)
    except Exception as e:
        fail_spinup.append(gdir.rgi_id)
        print("Spin-up失败: {} - {}".format(gdir.rgi_id, str(e)[:50]))

print("Spin-up成功: {}/{}".format(len(success_spinup), len(gdirs)))

In [ ]:
from oggm.core.massbalance import MonthlyTIModel, MultipleFlowlineMassBalance
cfg.PARAMS['evolution_model'] = 'FluxBased'
for gdir in gdirs:
    mb_model = MultipleFlowlineMassBalance(
        gdir,
        mb_model_class=MonthlyTIModel,
        filename='gcm_data',
        input_filesuffix='_LMR',
        melt_f=5.0,
        temp_bias=0.0,
        prcp_fac=2.5,
        check_calib_params=False,
        use_inversion_flowlines=True    # 添加这个
    )

    tasks.run_from_climate_data(
        gdir,
        mb_model=mb_model,
        ys=1640,
        ye=2000,
        output_filesuffix='_LIA_forward'
    )

In [ ]:
%%sql


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 1. 提取所有冰川结果
results = []
all_ds = {}


In [ ]:

for gdir in gdirs:
    fpath = gdir.get_filepath('model_diagnostics', filesuffix='_LIA_forward')
    try:
        ds = xr.open_dataset(fpath)
        all_ds[gdir.rgi_id] = ds

        vol_start = float(ds['volume_m3'].isel(time=0))
        vol_end = float(ds['volume_m3'].isel(time=-1))
        area_start = float(ds['area_m2'].isel(time=0))
        area_end = float(ds['area_m2'].isel(time=-1))
        len_start = float(ds['length_m'].isel(time=0))
        len_end = float(ds['length_m'].isel(time=-1))

        row = {
            'rgi_id': gdir.rgi_id,
            'volume_1640': vol_start,
            'volume_2000': vol_end,
            'area_1640': area_start,
            'area_2000': area_end,
            'length_1640': len_start,
            'length_2000': len_end,
        }
        results.append(row)
    except FileNotFoundError:
        print(f"文件不存在: {gdir.rgi_id}")
    except KeyError as ke:
        print(f"变量缺失 {gdir.rgi_id}: {ke}")

df_results = pd.DataFrame(results)

# 计算变化
df_results['vol_chg_pct'] = (df_results['volume_2000'] - df_results['volume_1640']) / df_results['volume_1640'] * 100
df_results['area_chg_pct'] = (df_results['area_2000'] - df_results['area_1640']) / df_results['area_1640'] * 100
df_results['len_chg_m'] = df_results['length_2000'] - df_results['length_1640']

print("=" * 60)
print("成功提取 {} 条冰川结果".format(len(df_results)))
print("=" * 60)
print("\n汇总统计:")
print(df_results.describe())

# 2. 随机选择3条冰川绘图
rng = np.random.default_rng(42)
id_list = list(all_ds.keys())
n_sample = min(3, len(id_list))
sample_ids = rng.choice(id_list, size=n_sample, replace=False)

fig, axes = plt.subplots(3, 3, figsize=(14, 10))

for i, rid in enumerate(sample_ids):
    ds = all_ds[rid]
    t = ds['time'].values

    # 体积
    vol = ds['volume_m3'].values / 1e9
    axes[0, i].plot(t, vol, 'b-')
    axes[0, i].set_title(rid)
    axes[0, i].set_ylabel('Volume (km³)')
    axes[0, i].grid(True, alpha=0.3)

    # 面积
    area = ds['area_m2'].values / 1e6
    axes[1, i].plot(t, area, 'g-')
    axes[1, i].set_ylabel('Area (km²)')
    axes[1, i].grid(True, alpha=0.3)

    # 长度
    length = ds['length_m'].values / 1000
    axes[2, i].plot(t, length, 'r-')
    axes[2, i].set_ylabel('Length (km)')
    axes[2, i].set_xlabel('Year')
    axes[2, i].grid(True, alpha=0.3)

fig.suptitle('LIA Forward Simulation (1640-2000): 3 Random Glaciers', fontsize=14)
fig.tight_layout()
fig.savefig('random_3_glaciers_timeseries.png', dpi=150)
plt.show()

# 3. 所有冰川汇总
vol_arrays = []
area_arrays = []
time_vals = None

for rid, ds in all_ds.items():
    if time_vals is None:
        time_vals = ds['time'].values
    vol_arrays.append(ds['volume_m3'].values)
    area_arrays.append(ds['area_m2'].values)

total_vol = np.sum(np.array(vol_arrays), axis=0)
total_area = np.sum(np.array(area_arrays), axis=0)

fig2, ax2 = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax2[0].plot(time_vals, total_vol / 1e9, 'b-', linewidth=2)
ax2[0].set_ylabel('Total Volume (km³)')
ax2[0].set_title('All {} Glaciers: Total Volume and Area (1640-2000)'.format(len(all_ds)))
ax2[0].grid(True, alpha=0.3)

ax2[1].plot(time_vals, total_area / 1e6, 'g-', linewidth=2)
ax2[1].set_ylabel('Total Area (km²)')
ax2[1].set_xlabel('Year')
ax2[1].grid(True, alpha=0.3)

fig2.tight_layout()
fig2.savefig('all_glaciers_total_timeseries.png', dpi=150)
plt.show()

# 打印汇总
v0, v1 = total_vol[0]/1e9, total_vol[-1]/1e9
a0, a1 = total_area[0]/1e6, total_area[-1]/1e6
v_pct = (total_vol[-1] - total_vol[0]) / total_vol[0] * 100
a_pct = (total_area[-1] - total_area[0]) / total_area[0] * 100

print("\n总量变化:")
print("体积: {:.2f} km³ (1640) → {:.2f} km³ (2000), 变化 {:.1f}%".format(v0, v1, v_pct))
print("面积: {:.2f} km² (1640) → {:.2f} km² (2000), 变化 {:.1f}%".format(a0, a1, a_pct))

In [ ]:
gdirs

In [ ]:
main_rgi_ids = lia_matched['MainRGI'].tolist()
rgi_to_calibrate = rgi[rgi['RGIId'].isin(main_rgi_ids)].copy()
import os
os.makedirs("/mnt/f/oggm_related/rgi_for_calibration", exist_ok=True)
rgi_to_calibrate.to_file("/mnt/f/oggm_related/rgi_for_calibration/rgi_245_for_calib.shp")
print("已保存: {} 条".format(len(rgi_to_calibrate)))

In [ ]:
from oggm import cfg, workflow, tasks

cfg.initialize()
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_multiprocessing'] = True
cfg.PATHS['working_dir'] = '/mnt/f/oggm_related/oggm_rgi_calibration'

In [ ]:
from oggm import GlacierDirectory

gdirs_rgi = []
for idx, row in rgi_to_calibrate.iterrows():
    try:
        gdir = GlacierDirectory(row, reset=True)
        gdirs_rgi.append(gdir)
    except Exception as e:
        print("失败: {} - {}".format(row['RGIId'], str(e)[:50]))

print("初始化完成: {}/{}".format(len(gdirs_rgi), len(rgi_to_calibrate)))

In [ ]:
gdir = gdirs[0]
print(gdir.rgi_id)
print(gdir.read_pickle('glacier_region'))  # 或者
print(gdir.get_diagnostics())